# NestedKriging on the Branin 2D function (Octave)

This notebook demonstrates **NestedKriging**, a divide-and-conquer Gaussian Process for
large designs: the data are partitioned into $p$ groups, one Kriging submodel is fitted
per group (all sharing a **common prior** after hyperparameter unification), and
predictions are aggregated.

Available aggregations:
- **NK** (default): the optimal aggregation of Rullière, Durrande, Bachoc & Chevalier
  (*Statistics & Computing*, 2018) — itself a kriging predictor: it interpolates the
  design and provides consistent variances;
- **PoE / gPoE / BCM / rBCM**: precision-weighted products of experts (cheaper, but
  no interpolation guarantee).

Fit cost drops from $O(n^3)$ to $O(n^3/p^2)$ per likelihood evaluation.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 800$)
4. Fit `NestedKriging` and inspect the partition
5. Predict on a grid: mean, stdev, RMSE, interpolation check
6. Compare aggregations
7. Warped variant (WarpKriging submodels)


## 0. Setup

Build the C++ core and the Octave binding from source (skip if already built).
Requires: `cmake`, a C++ compiler, Octave ≥ 6.0.

In [ ]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

## 2. Branin function

In [ ]:
function z = branin(X)
    x1 = X(:,1) * 15 - 5;
    x2 = X(:,2) * 15;
    z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
        + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
end

grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);

figure; contourf(G1, G2, z_true, 20); colorbar; title('True Branin');

## 3. Design of experiments (n = 800)

In [ ]:
rand('seed', 123);
n = 800;
X = rand(n, 2);
y = branin(X);
size(X)

## 4. Fit NestedKriging (NK aggregation, 8 groups)

In [ ]:
nk = NestedKriging(y, X, "matern5_2", 8);  % aggregation="NK", partition="kmeans" by default
disp(nk.summary());

## 5. Prediction on the grid

In [ ]:
[p_mean, p_stdev] = nk.predict(grid_pts);

figure; contourf(G1, G2, reshape(p_mean, 50, 50), 20); colorbar; title('NK mean');
figure; contourf(G1, G2, reshape(p_stdev, 50, 50), 20); colorbar; title('NK stdev');

printf('RMSE vs true Branin : %g\n', sqrt(mean((p_mean - z_true(:)).^2)));
m_X = nk.predict(X);
printf('max |mean - y| at design points (interpolation): %g\n', max(abs(m_X - y)));

## 6. Comparing aggregations

In [ ]:
aggs = {"NK", "gPoE", "rBCM", "PoE", "BCM"};
for i = 1:numel(aggs)
    k = NestedKriging(y, X, "matern5_2", 8, aggs{i});
    m = k.predict(grid_pts);
    printf('%-5s RMSE = %.4f\n', aggs{i}, sqrt(mean((m - z_true(:)).^2)));
end

## 7. Warped variant

All positional arguments up to `parameters` (pass `[]`) must be given to reach the
`warping` cell array.

In [ ]:
nkw = NestedKriging(y, X, "gauss", 8, "NK", "kmeans", 123, ...
                    "constant", "BFGS", "LL", [], {"kumaraswamy", "kumaraswamy"});
mw = nkw.predict(grid_pts);
printf('RMSE (warped) = %.4f\n', sqrt(mean((mw - z_true(:)).^2)));

## Notes

- The NK aggregation **interpolates** whatever the hyperparameters: it is a property of
  the aggregation itself, not of the fit.
- NK requires a **constant trend**; use the PoE family for other trends.
- Memory/speed of the NK prediction can be tuned with `set_predict_chunk`.
- In the warped case, the prior hyperparameters $(\theta, \text{warp})$ are estimated by a
  **single reference fit on a global subsample** (default 1000 points, see
  `set_warp_subsample`), then submodels are fitted in closed form (`optim="none"`).
